In [12]:
import pandas as pd
from pathlib import Path

# Load the master dataset from the Gold layer
gold_path = Path("data/03_gold/ecosystem_master.parquet")
df_master = pd.read_parquet(gold_path)

print(f"✅ df_master loaded! Total rows: {len(df_master)}")

✅ df_master loaded! Total rows: 27782


# Export to GEPHI

In [13]:
import pandas as pd
import networkx as nx
from pathlib import Path

# Define directories
gold_dir = Path("data/03_gold")
viz_dir = Path("viz/snapshots")

# Ensure the output directory exists
viz_dir.mkdir(parents=True, exist_ok=True)

def generate_clean_gexf(snapshot_key):
    edge_file = gold_dir / f"gold_edges_{snapshot_key}.parquet"
    node_file = gold_dir / f"gold_nodes_{snapshot_key}.parquet"
    
    # Quick sanity check to ensure the files for this date exist before reading
    if not edge_file.exists() or not node_file.exists():
        print(f"⚠️ Skipping snapshot {snapshot_key}: Parquet files not found.")
        return

    # Load strictly from Gold
    edges_df = pd.read_parquet(edge_file)
    nodes_df = pd.read_parquet(node_file)
    
    # Build graph object
    G = nx.from_pandas_edgelist(edges_df, source='source_node', target='target_node', create_using=nx.DiGraph())
    G.add_nodes_from(nodes_df['node_name'].tolist())
    
    # Map attributes safely
    for _, row in nodes_df.iterrows():
        node = row['node_name']
        if G.has_node(node):
            G.nodes[node]['pagerank'] = float(row['pagerank'])
            G.nodes[node]['in_degree'] = int(row['in_degree'])
            G.nodes[node]['is_in_bcc'] = str(row['is_in_bcc'])
            
    # Export using the verified directory path
    out_file = viz_dir / f"crystal_presentation_{snapshot_key}.gexf"
    nx.write_gexf(G, out_file)
    print(f"✅ Generated final Gephi asset: {out_file.name}")


# --- Generate Timeline and Process Snapshots ---

# 1. Anchor to your verified database state
anchor_date = "2026-05-06"

# 2. Generate exactly 30 snapshots spaced 100 days apart ending on the anchor
snapshot_dates = pd.date_range(end=anchor_date, periods=30, freq="100D")

print(f"Processing 30 snapshots from {snapshot_dates[0].strftime('%Y-%m-%d')} to {snapshot_dates[-1].strftime('%Y-%m-%d')}...\n")

# 3. Loop over the generated timestamps, format them as strings, and build the GEXF assets
for ts in snapshot_dates:
    snapshot_key = ts.strftime('%Y-%m-%d')
    generate_clean_gexf(snapshot_key)

print("\n🏁 All available network slices have been written to disk!")

Processing 30 snapshots from 2018-05-28 to 2026-05-06...

✅ Generated final Gephi asset: crystal_presentation_2018-05-28.gexf
✅ Generated final Gephi asset: crystal_presentation_2018-09-05.gexf
✅ Generated final Gephi asset: crystal_presentation_2018-12-14.gexf
✅ Generated final Gephi asset: crystal_presentation_2019-03-24.gexf
✅ Generated final Gephi asset: crystal_presentation_2019-07-02.gexf
✅ Generated final Gephi asset: crystal_presentation_2019-10-10.gexf
✅ Generated final Gephi asset: crystal_presentation_2020-01-18.gexf
✅ Generated final Gephi asset: crystal_presentation_2020-04-27.gexf
✅ Generated final Gephi asset: crystal_presentation_2020-08-05.gexf
✅ Generated final Gephi asset: crystal_presentation_2020-11-13.gexf
✅ Generated final Gephi asset: crystal_presentation_2021-02-21.gexf
✅ Generated final Gephi asset: crystal_presentation_2021-06-01.gexf
✅ Generated final Gephi asset: crystal_presentation_2021-09-09.gexf
✅ Generated final Gephi asset: crystal_presentation_2021-1

In [14]:
import pandas as pd
import networkx as nx
from pathlib import Path

gold_dir = Path("data/03_gold")
viz_dir = Path("viz")
viz_dir.mkdir(parents=True, exist_ok=True)

# 1. Recreate our chronological timeline sequence
anchor_date = "2026-05-06"
snapshot_dates = pd.date_range(end=anchor_date, periods=30, freq="100D")
sorted_snapshot_keys = [ts.strftime('%Y-%m-%d') for ts in snapshot_dates]

# Structures to track the first and last appearance of every element
node_meta = {}  # node_name -> {'start': str, 'end': str, 'pagerank': float, 'in_degree': int, 'is_in_bcc': str}
edge_meta = {}  # (source, target) -> {'start': str, 'end': str}

print("Analyzing 30 snapshots to build a temporal map...")

for idx, snapshot_key in enumerate(sorted_snapshot_keys):
    edge_file = gold_dir / f"gold_edges_{snapshot_key}.parquet"
    node_file = gold_dir / f"gold_nodes_{snapshot_key}.parquet"
    
    if not edge_file.exists() or not node_file.exists():
        continue
        
    edges_df = pd.read_parquet(edge_file)
    nodes_df = pd.read_parquet(node_file)
    
    # Track Nodes
    for _, row in nodes_df.iterrows():
        node = row['node_name']
        if node not in node_meta:
            # First time seeing this shard! Set its start date
            node_meta[node] = {
                'start': snapshot_key,
                'end': anchor_date, # Default to alive until the end
                'pagerank': float(row['pagerank']),
                'in_degree': int(row['in_degree']),
                'is_in_bcc': str(row['is_in_bcc'])
            }
        else:
            # Shard is still here, update its latest metrics
            node_meta[node]['pagerank'] = float(row['pagerank'])
            node_meta[node]['in_degree'] = int(row['in_degree'])
            node_meta[node]['is_in_bcc'] = str(row['is_in_bcc'])

    # Track Edges (Dependencies)
    for _, row in edges_df.iterrows():
        edge = (row['source_node'], row['target_node'])
        if edge not in edge_meta:
            edge_meta[edge] = {
                'start': snapshot_key,
                'end': anchor_date
            }

# 2. Assemble the Master Directed Graph
G_master = nx.DiGraph()

# Add nodes with temporal spells and final metadata
for node, attrs in node_meta.items():
    G_master.add_node(
        node,
        label=node,
        start=attrs['start'],
        end=attrs['end'],
        pagerank=attrs['pagerank'],
        in_degree=attrs['in_degree'],
        is_in_bcc=attrs['is_in_bcc']
    )

# Add edges with temporal spells
for (source, target), attrs in edge_meta.items():
    # Only add the edge if both bounding nodes exist in our metadata universe
    if G_master.has_node(source) and G_master.has_node(target):
        G_master.add_edge(
            source, 
            target, 
            start=attrs['start'], 
            end=attrs['end']
        )

# 3. Export to a dynamic GEXF
output_file = viz_dir / "crystal_ecosystem_evolution.gexf"
nx.write_gexf(G_master, output_file, version="1.2draft")

print(f"\n🏁 Success! Master timeline graph written to: {output_file}")
print(f"Total historical nodes: {G_master.number_of_nodes()}")
print(f"Total historical edges: {G_master.number_of_edges()}")

Analyzing 30 snapshots to build a temporal map...

🏁 Success! Master timeline graph written to: viz/crystal_ecosystem_evolution.gexf
Total historical nodes: 1165
Total historical edges: 1897
